# OffsideX — v1: Initial Automated Offside Baseline

- **Pitch Segmentation:** HSV turf threshold + largest convex hull + `15x15` erosion.
- **Detection & Pose:** Single-pass `YOLOv8n-Pose` at `imgsz=640`.
- **Team Split & Decision:** `KMeans(n_clusters=3)` on mean RGB torso crops + raw 2D X comparison.


In [ ]:
!pip install -q ultralytics scikit-learn opencv-python-headless

from pathlib import Path

import cv2
import numpy as np
from google.colab import drive
from sklearn.cluster import KMeans
from ultralytics import YOLO

drive.mount("/content/drive")
DATASET_DIR = Path("/content/drive/MyDrive/Offside_Detector/dataset")

pose_model = YOLO("yolov8n-pose.pt")
GRASS_LOWER_V1 = np.array([30, 40, 40], dtype=np.uint8)  # HSV bounds for pitch turf
GRASS_UPPER_V1 = np.array([85, 255, 255], dtype=np.uint8)


In [ ]:
def get_pitch_mask_v1(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    raw = cv2.inRange(hsv, GRASS_LOWER_V1, GRASS_UPPER_V1)
    closed = cv2.morphologyEx(raw, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask = np.zeros((h, w), dtype=np.uint8)
    if contours:
        hull = cv2.convexHull(max(contours, key=cv2.contourArea))
        cv2.drawContours(mask, [hull], -1, 255, thickness=cv2.FILLED)
    return cv2.erode(mask, np.ones((15, 15), dtype=np.uint8), iterations=1)  # Trim noisy mask perimeter


def detect_players_direct_v1(img: np.ndarray, pitch_mask: np.ndarray):
    h, w = img.shape[:2]
    res = pose_model(img, conf=0.10, imgsz=640, verbose=False)[0]  # Single-pass pose inference
    players = []
    if res.boxes is None or res.keypoints is None:
        return players

    for box, kpts, kconf in zip(
        res.boxes.xyxy.cpu().numpy(),
        res.keypoints.xy.cpu().numpy(),
        res.keypoints.conf.cpu().numpy(),
    ):
        x1, y1, x2, y2 = box.astype(int)
        cx = int(np.clip((x1 + x2) / 2, 0, w - 1))
        cy = int(np.clip((y1 + y2) / 2, 0, h - 1))
        fy = int(np.clip(y2 - 2, 0, h - 1))

        if pitch_mask[fy, cx] == 0 and pitch_mask[cy, cx] == 0:
            continue

        valid = [kpts[i, 0] for i in range(17) if kconf[i] > 0.25]  # Rightmost active keypoint
        max_x = float(max(valid)) if valid else float(x2)

        bw, bh = max(1, x2 - x1), max(1, y2 - y1)
        crop = img[
            max(0, y1 + int(0.15 * bh)) : min(h, y1 + int(0.55 * bh)),
            max(0, x1 + int(0.20 * bw)) : min(w, x1 + int(0.80 * bw)),
        ]
        if crop.size == 0:
            continue
        mean_rgb = np.mean(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).reshape(-1, 3), axis=0)
        players.append({"box": box, "max_x": max_x, "color": mean_rgb})
    return players


def predict_raw_2d_v1(img: np.ndarray):
    h, w = img.shape[:2]
    mask = get_pitch_mask_v1(img)
    players = detect_players_direct_v1(img, mask)
    if len(players) < 4:
        raise ValueError(f"Only {len(players)} players detected at imgsz=640.")

    feats = np.array([p["color"] for p in players], dtype=np.float32)
    labels = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(feats)  # Cluster teams and referee

    c0 = [p for p, lbl in zip(players, labels) if lbl == 0]
    c1 = [p for p, lbl in zip(players, labels) if lbl == 1]
    if not c0 or not c1:
        raise ValueError("KMeans-3 collapsed clusters.")

    attackers, defenders = c0, c1
    rightmost_def = max(defenders, key=lambda p: p["max_x"])
    if rightmost_def["max_x"] > 0.75 * w and len(defenders) >= 2:  # Exclude far-right goalkeeper
        defenders = [p for p in defenders if p is not rightmost_def]

    last_def_x = max(p["max_x"] for p in defenders)
    lead_att_x = max(p["max_x"] for p in attackers)
    raw_diff_x = (lead_att_x - last_def_x) / w
    return ("OFFSIDE" if raw_diff_x > 0 else "ONSIDE"), raw_diff_x


In [ ]:
FOLDERS = [
    ("offside", "OFFSIDE", False),
    ("offside/left", "OFFSIDE", True),
    ("onside", "ONSIDE", False),
    ("onside/left", "ONSIDE", True),
]
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

processed, skipped, correct = 0, 0, 0
for rel_dir, label, flip_left in FOLDERS:
    folder = DATASET_DIR / rel_dir
    if not folder.exists():
        continue
    for img_path in sorted(p for p in folder.iterdir() if p.suffix.lower() in EXTS):
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
        if flip_left:
            img = cv2.flip(img, 1)  # Normalize attack direction
        try:
            pred, _ = predict_raw_2d_v1(img)
        except ValueError:
            skipped += 1
            continue
        processed += 1
        if pred == label:
            correct += 1

acc = (100.0 * correct / processed) if processed else 0.0
print(f"Processed: {processed} | Skipped: {skipped} | Accuracy: {acc:.2f}% ({correct}/{processed})")
